# 07 — Empirical width bottleneck

Controlled endpoint-only experiment for configuration (c). The student architecture, corpus, optimizer, gauge updates, and evaluation protocol remain fixed while the active rank of the PCA teacher interface varies.

The notebook reports retained teacher energy, normalized cosine-Gram distortion, its rank-constrained lower bound, and downstream performance. The lower ranks are zero-padded to the native 384-dimensional student space before Procrustes alignment, so this sweep changes the retained teacher subspace rather than the student capacity.

In [ ]:
# 1. Cấu hình thí nghiệm
from pathlib import Path

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
AUTO_PULL_REPO, INSTALL_REQUIREMENTS = True, True
PAIR = "qwen3_0.6b_to_minilm_h384"
TRAIN_DATA_REL = Path("data/train_set/merged_3_data_5k_each.csv")
RANKS = [64, 128, 256, 384]
SEEDS = [42, 43, 44]
STUDENT_DIM = 384
BATCH_SIZE, EPOCHS, LR = 128, 5, 7e-5
MAX_LENGTH, NUM_WORKERS = 256, 2
GEOMETRY_ROWS, GEOMETRY_SEED = 2048, 0
EXECUTE, STOP_ON_ERROR, REQUIRE_COMPLETE = True, True, True
RENDER_FIGURE, COPY_TO_PAPER = True, False
SAVE_TO_GOOGLE_DRIVE = False
CUDA_VISIBLE_DEVICES = "0"
MAX_PARALLEL_JOBS, GPUS = 3, None
RUN_NAME_OVERRIDE = "analysis_width_bottleneck_qwen06b_minilm384_15k_v1"
RUN_NAME = RUN_NAME_OVERRIDE

assert RANKS == sorted(set(RANKS)) and min(RANKS) > 0
assert max(RANKS) == STUDENT_DIM
assert SEEDS and len(SEEDS) == len(set(SEEDS))
print(f"Pair: {PAIR}; ranks: {RANKS}; seeds: {SEEDS}")
print(f"Run: {RUN_NAME}")

In [ ]:
# 2. Dùng repo hiện tại hoặc clone trên Colab; cài dependencies
import subprocess
import sys

cwd = Path.cwd().resolve()
PROJECT_DIR = next((p for p in (cwd, cwd.parent) if (p / "main.py").is_file()), None)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"

tracked = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"],
    check=True, capture_output=True, text=True,
).stdout.strip()
if AUTO_PULL_REPO and not tracked:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif AUTO_PULL_REPO:
    print("[git] Bỏ qua pull vì repo có tracked changes.")
if INSTALL_REQUIREMENTS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
        check=True,
    )
git_head = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
sys.path[:0] = [str(PROJECT_DIR), str(PROJECT_DIR / "notebooks")]
print(f"Repo: {PROJECT_DIR} @ {git_head}")

In [ ]:
# 3. Output, GPU và dữ liệu
import torch

try:
    from google.colab import drive as colab_drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    colab_drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/embedding-kd-runs")
else:
    OUTPUT_BASE = PROJECT_DIR / "runs"

RUN_ROOT = OUTPUT_BASE / RUN_NAME
CACHE_DIR = OUTPUT_BASE / "teacher_cache"
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"
if EXECUTE:
    assert torch.cuda.is_available(), "Hãy bật GPU runtime trước khi train."
    if hasattr(torch.cuda, "is_bf16_supported"):
        assert torch.cuda.is_bf16_supported(), "GPU phải hỗ trợ BF16."
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Training data: {TRAIN_DATA}")
print(f"Output root: {RUN_ROOT}")

In [ ]:
# 4. Tạo plan — một endpoint-only run cho mỗi (rank, seed)
import shlex
from _analysis_common import PAIRS, collect_jobs, geoode_command, run_jobs

pair = PAIRS[PAIR]
run_root = RUN_ROOT
jobs = []
for rank in RANKS:
    for seed in SEEDS:
        run_dir = run_root / f"rank_{rank}" / f"seed_{seed}"
        extra = [
            "--projection_type", "pca",
            "--projection_rank", rank,
            "--gauge_align",
            "--gauge_rotation", "procrustes",
            "--gauge_refit_every", 1,
            "--lambda_end", 1,
            "--lambda_ctr", 0,
            "--lambda_topo", 0,
            "--no_eval_retrieval",
        ]
        jobs.append({
            "name": f"rank_{rank}/seed_{seed}",
            "rank": rank,
            "seed": seed,
            "run_dir": run_dir,
            "command": geoode_command(
                PROJECT_DIR, pair=pair, train_data=TRAIN_DATA, cache_dir=CACHE_DIR,
                run_dir=run_dir, seed=seed, batch_size=BATCH_SIZE, epochs=EPOCHS,
                learning_rate=LR, max_length=MAX_LENGTH, num_workers=NUM_WORKERS,
                extra=extra,
            ),
        })
print(f"Plan: {len(RANKS)} ranks × {len(SEEDS)} seeds = {len(jobs)} jobs")
for job in jobs:
    print(shlex.join(job["command"]))

In [ ]:
# 5. Chạy jobs; final-test record là resume boundary
from IPython.display import display
from _analysis_common import prewarm_teacher_cache

if EXECUTE:
    prewarm_teacher_cache(
        PROJECT_DIR, pair=pair, train_data=TRAIN_DATA, cache_dir=CACHE_DIR,
        max_length=MAX_LENGTH, cuda_visible_devices=CUDA_VISIBLE_DEVICES,
    )
    display(run_jobs(
        PROJECT_DIR, jobs, cuda_visible_devices=CUDA_VISIBLE_DEVICES,
        stop_on_error=STOP_ON_ERROR, max_parallel=MAX_PARALLEL_JOBS, gpus=GPUS,
    ))
else:
    print("Dry run: đặt EXECUTE=True để chạy các job còn thiếu.")

In [ ]:
# 6. Tổng hợp geometry và downstream performance
import numpy as np
import pandas as pd
import torch.nn.functional as F
from _analysis_common import load_teacher_cache, teacher_cache_path
from src import structural_audit as audit

results = collect_jobs(jobs)
results.to_csv(run_root / "width_bottleneck_by_run.csv", index=False)
done = results.query("status == 'done'").copy()
expected = {rank: len(SEEDS) for rank in RANKS}
counts = done.groupby("rank").size().to_dict() if not done.empty else {}
if REQUIRE_COMPLETE and counts != expected:
    raise RuntimeError(f"Incomplete rank grid: got {counts}, expected {expected}")

score_summary = done.groupby("rank", as_index=False).agg(
    avg_iod_mean=("avg_iod", "mean"), avg_iod_std=("avg_iod", "std"),
    avg_ood_mean=("avg_ood", "mean"), avg_ood_std=("avg_ood", "std"),
    avg_mean=("avg_all", "mean"), avg_std=("avg_all", "std"),
    n=("avg_all", "count"),
)

cache_path = teacher_cache_path(
    PROJECT_DIR, CACHE_DIR, pair=pair, train_data=TRAIN_DATA, max_length=MAX_LENGTH,
)
teacher, _ = load_teacher_cache(cache_path)
rng = np.random.default_rng(GEOMETRY_SEED)
indices = np.sort(rng.choice(len(teacher), min(GEOMETRY_ROWS, len(teacher)), replace=False))
teacher_sample = F.normalize(teacher[torch.as_tensor(indices)].float(), dim=-1)
teacher_gram = teacher_sample @ teacher_sample.T
gram_scale = teacher_gram.square().sum().clamp_min(1e-12)
singular_values = torch.linalg.svdvals(teacher_sample)
spectrum_scale = singular_values.pow(4).sum().clamp_min(1e-12)

geometry_rows = []
for rank in RANKS:
    job = next(job for job in jobs if job["rank"] == rank and job["seed"] == SEEDS[0])
    saved = audit.load_saved_projection(job["run_dir"] / "teacher_projection.pt")
    targets = audit.targets_from_saved(teacher_sample, saved)
    target_gram = targets @ targets.T
    gram_distortion = float((teacher_gram - target_gram).square().sum() / gram_scale)
    lower_bound = float(singular_values[rank:].pow(4).sum() / spectrum_scale)
    if gram_distortion + 1e-5 < lower_bound:
        raise RuntimeError(
            f"Rank-{rank} distortion {gram_distortion:.6f} violates lower bound {lower_bound:.6f}"
        )
    geometry_rows.append({
        "rank": rank,
        "retained_energy": float(saved["explained_energy"]),
        "gram_lower_bound": lower_bound,
        "target_gram_distortion": gram_distortion,
    })
geometry_summary = pd.DataFrame(geometry_rows)
summary = geometry_summary.merge(score_summary, on="rank", validate="one_to_one")
summary.to_csv(run_root / "width_bottleneck_summary.csv", index=False)
display(summary.round(5))

In [ ]:
# 7. Figure paper-ready và manifest
import json
import shutil
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.2,
    "axes.titlesize": 7.5, "axes.labelsize": 7.2,
    "xtick.labelsize": 6.6, "ytick.labelsize": 6.6,
    "axes.linewidth": 0.65, "pdf.fonttype": 42, "ps.fonttype": 42,
    "savefig.dpi": 300,
})
figure_files = []
if RENDER_FIGURE:
    fig, axes = plt.subplots(1, 3, figsize=(5.5, 2.05))
    x = summary["rank"].to_numpy()
    axes[0].plot(x, 100 * summary["retained_energy"], "o-", color="#2A78D6", lw=1.4, ms=3.5)
    axes[0].set_title("(a) Retained energy")
    axes[0].set_ylabel("Teacher energy (%)")

    axes[1].plot(x, 100 * summary["gram_lower_bound"], "--", color="#7A7A7A", lw=1.3, label="Rank floor")
    axes[1].plot(x, 100 * summary["target_gram_distortion"], "o-", color="#E07A35", lw=1.4, ms=3.5, label="PCA target")
    axes[1].set_title("(b) Gram distortion")
    axes[1].set_ylabel("Normalized error (%)")
    axes[1].legend(frameon=False, fontsize=6.2, handlelength=1.5)

    axes[2].errorbar(
        x, 100 * summary["avg_mean"], yerr=100 * summary["avg_std"],
        fmt="o-", color="#2A9D6F", lw=1.4, ms=3.5, capsize=2,
    )
    axes[2].set_title("(c) Downstream utility")
    axes[2].set_ylabel("Avg. score")

    for axis in axes:
        axis.set_xlabel(r"Retained rank $k$")
        axis.set_xticks(RANKS)
        axis.grid(alpha=0.18, linewidth=0.5)
        axis.spines[["top", "right"]].set_visible(False)
    fig.subplots_adjust(left=0.09, right=0.99, bottom=0.23, top=0.84, wspace=0.52)
    figure_dir = run_root / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    for suffix in ("pdf", "png"):
        path = figure_dir / f"width_bottleneck.{suffix}"
        fig.savefig(path, facecolor="white")
        figure_files.append(path)
        print(path)
    plt.show()
    if COPY_TO_PAPER:
        paper_dir = PROJECT_DIR / "docs" / "latex_iclr" / "figures"
        for path in figure_files:
            shutil.copy2(path, paper_dir / path.name)

manifest = {
    "git_head": git_head, "pair": PAIR, "train_data": str(TRAIN_DATA_REL),
    "ranks": RANKS, "seeds": SEEDS, "student_dim": STUDENT_DIM,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "learning_rate": LR,
    "objective": "endpoint_only", "gauge_refit_every": 1,
    "geometry_rows": len(teacher_sample), "geometry_seed": GEOMETRY_SEED,
    "figures": [str(path) for path in figure_files],
}
(run_root / "experiment_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8",
)